# OxigraphExplorer Guide

This notebook demonstrates how to use `OxigraphExplorer`, a pyoxigraph-backed RDF graph explorer for the AI Atlas Nexus knowledge graph.

## Overview

`OxigraphExplorer` provides:
- **Drop-in compatibility** with `AtlasExplorer` — returns typed Pydantic objects
- **Efficient graph queries** via an indexed RDF store
- **Raw SPARQL access** for advanced queries
- **Hybrid architecture** — RDF store for traversal, Pydantic objects for data

## Setup

In [1]:
from ai_atlas_nexus import AIAtlasNexus
from ai_atlas_nexus.blocks.atlas_explorer import OxigraphExplorer, AtlasExplorer

# Load the ontology
nexus = AIAtlasNexus()
print(f"Loaded ontology with {len(nexus.get_all_risks())} risks")

/Users/ingevejs/Documents/workspace/ingelise/risk-atlas-nexus/v-ai-atlas-nexus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-04-23 16:52:18:729] - INFO - AIAtlasNexus - Created AIAtlasNexus instance. Base_dir: None


Loaded ontology with 546 risks


## Part 1: Using OxigraphExplorer Standalone

You can create an `OxigraphExplorer` instance directly from a Container object:

In [2]:
# Create OxigraphExplorer instance
ox = OxigraphExplorer(nexus._ontology)
print("OxigraphExplorer initialized")

# Compare with AtlasExplorer
atlas = AtlasExplorer(nexus._ontology)
print("AtlasExplorer initialized")

OxigraphExplorer initialized
AtlasExplorer initialized


### Basic Queries

In [3]:
# Get all classes in the knowledge graph
classes = ox.get_all_classes()
print(f"Available classes ({len(classes)}): {classes[:5]}...")

Available classes (22): ['riskincidents', 'vocabularies', 'datasets', 'aimodels', 'llmintrinsics']...


In [4]:
# Get all risks by collection key
risks = ox.get_all("risks")
print(f"Retrieved {len(risks)} risks via collection key 'risks'")

# Show first risk (returns typed Pydantic object)
first_risk = risks[0]
print(f"\nFirst risk:")
print(f"  Type: {type(first_risk).__name__}")
print(f"  ID: {first_risk.id}")
print(f"  Name: {first_risk.name}")
print(f"  Taxonomy: {first_risk.isDefinedByTaxonomy}")

Retrieved 546 risks via collection key 'risks'

First risk:
  Type: Risk
  ID: atlas-evasion-attack
  Name: Evasion attack
  Taxonomy: ibm-risk-atlas


In [5]:
# You can also request by class name (singular or plural)
risks_by_class = ox.get_all("Risk")
risks_by_class_plural = ox.get_all("Risks")

print(f"Risks by class name 'Risk': {len(risks_by_class)}")
print(f"Risks by class name 'Risks': {len(risks_by_class_plural)}")
print(f"Match collection key results: {len(risks_by_class) == len(risks) and len(risks_by_class_plural) == len(risks)}")

Risks by class name 'Risk': 546
Risks by class name 'Risks': 546
Match collection key results: True


In [6]:
# Get all actions
actions = ox.get_all("actions")
actions_by_class = ox.get_all("Action")

print(f"Retrieved {len(actions)} actions via collection key")
print(f"Retrieved {len(actions_by_class)} actions via class name")
print(f"Match: {len(actions) == len(actions_by_class)}")

Retrieved 1085 actions via collection key
Retrieved 1085 actions via class name
Match: True


In [7]:
# Get by ID
risk_id = risks[0].id
retrieved = ox.get_by_id(None, risk_id)

print(f"Retrieved risk by ID: {retrieved.id}")
print(f"  Name: {retrieved.name}")
print(f"  Match: {retrieved.id == risk_id}")

Retrieved risk by ID: atlas-evasion-attack
  Name: Evasion attack
  Match: True


In [8]:
# Filter by attribute
nist_actions = ox.get_by_attribute("actions", "isDefinedByTaxonomy", "nist-ai-rmf")
print(f"Actions from NIST AI RMF taxonomy: {len(nist_actions)}")

if nist_actions:
    print(f"\nFirst NIST action:")
    print(f"  ID: {nist_actions[0].id}")
    print(f"  Name: {nist_actions[0].name}")

Actions from NIST AI RMF taxonomy: 212

First NIST action:
  ID: MG-4.3-003
  Name: MG-4.3-003


In [9]:
# Filter by multiple criteria (AND logic)
filtered = ox.filter_instances(
    "risks",
    {
        "isDefinedByTaxonomy": "ibm-risk-atlas",
    }
)
print(f"Filtered risks (IBM Risk Atlas): {len(filtered)}")

Filtered risks (IBM Risk Atlas): 99


### Advanced: Raw SPARQL Queries

The `sparql_query()` method gives you direct access to the RDF graph:

In [10]:
# Find all risks in the knowledge graph via SPARQL
sparql = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX nexus: <https://ibm.github.io/ai-atlas-nexus/ontology/>

SELECT ?s WHERE {
    ?s rdf:type nexus:Risk .
}
LIMIT 5
"""

results = ox.sparql_query(sparql)
print(f"First 5 risks (raw SPARQL results):")
for r in results:
    uri = r['?s']
    # Extract ID from URI
    risk_id = uri.split('/')[-1].rstrip('>')
    print(f"  {risk_id}")

First 5 risks (raw SPARQL results):
  credo-risk-049
  credo-risk-048
  credo-risk-047
  credo-risk-046
  credo-risk-045


In [11]:
# Find relationships: which actions target which risks?
sparql_relationships = """
PREFIX nexus: <https://ibm.github.io/ai-atlas-nexus/ontology/>

SELECT ?action ?risk WHERE {
    ?action nexus:hasRelatedRisk ?risk .
}
LIMIT 5
"""

relationships = ox.sparql_query(sparql_relationships)
print(f"Action-Risk relationships (sample):")
for rel in relationships:
    action_id = rel['?action'].split('/')[-1].rstrip('>')
    risk_id = rel['?risk'].split('/')[-1].rstrip('>')
    print(f"  {action_id} -> {risk_id}")

Action-Risk relationships (sample):
  credo-act-control-042 -> credo-risk-046
  credo-act-control-041 -> credo-risk-046
  credo-act-control-040 -> credo-risk-046
  credo-act-control-037 -> credo-risk-009
  credo-act-control-037 -> credo-risk-003


## Part 2: Comparing OxigraphExplorer with the Library

The `AIAtlasNexus` library uses `AtlasExplorer` internally for optimization. You can use `OxigraphExplorer` directly for the same data with graph query capabilities:

In [12]:
# Compare library methods with direct explorer usage
ox = OxigraphExplorer(nexus._ontology)
atlas = AtlasExplorer(nexus._ontology)

print("Comparing risk retrieval methods:")
print()

# Library method
lib_risks = nexus.get_all_risks()
print(f"1. nexus.get_all_risks():")
print(f"   Returns {len(lib_risks)} risks")
print()

# Direct OxigraphExplorer
ox_risks = ox.get_all("risks")
print(f"2. OxigraphExplorer.get_all('risks'):")
print(f"   Returns {len(ox_risks)} risks")
print()

# Direct OxigraphExplorer by class name
ox_risks_class = ox.get_all("Risk")
print(f"3. OxigraphExplorer.get_all('Risk') [by class name]:")
print(f"   Returns {len(ox_risks_class)} risks")
print()

# Direct AtlasExplorer
atlas_risks = atlas.get_all("risks")
print(f"4. AtlasExplorer.get_all('risks'):")
print(f"   Returns {len(atlas_risks)} risks")
print()

# Verify results match for critical lookups
print("Verification - get_by_id consistency:")
test_id = lib_risks[0].id
ox_obj = ox.get_by_id(None, test_id)
atlas_obj = atlas.get_by_id(None, test_id)
lib_obj = nexus.get_risk(id=test_id)

print(f"  Risk ID: {test_id}")
print(f"  OxigraphExplorer found: {ox_obj is not None}")
print(f"  AtlasExplorer found: {atlas_obj is not None}")
print(f"  Library found: {lib_obj is not None}")
print(f"  All match: {ox_obj is not None and atlas_obj is not None and lib_obj is not None}")

Comparing risk retrieval methods:

1. nexus.get_all_risks():
   Returns 546 risks

2. OxigraphExplorer.get_all('risks'):
   Returns 546 risks

3. OxigraphExplorer.get_all('Risk') [by class name]:
   Returns 546 risks

4. AtlasExplorer.get_all('risks'):
   Returns 546 risks

Verification - get_by_id consistency:
  Risk ID: atlas-evasion-attack
  OxigraphExplorer found: True
  AtlasExplorer found: True
  Library found: True
  All match: True


## Part 3: Why Use OxigraphExplorer?
It could be suitable for adding more complex graph SPARQL queries

In [13]:
# Example: Complex graph query via SPARQL
# "Find all risks that are mitigated by actions in the NIST AI RMF"

sparql_complex = """
PREFIX nexus: <https://ibm.github.io/ai-atlas-nexus/ontology/>

SELECT ?risk ?action WHERE {
    ?action nexus:hasRelatedRisk ?risk .
    ?action nexus:isDefinedByTaxonomy "nist-ai-rmf" .
}
LIMIT 10
"""

print("Complex query: Risks mitigated by NIST AI RMF actions")
results = ox.sparql_query(sparql_complex)
print(f"Found {len(results)} risk-action pairs:")

for i, rel in enumerate(results[:3], 1):
    risk_id = rel['?risk'].split('/')[-1].rstrip('>')
    action_id = rel['?action'].split('/')[-1].rstrip('>')
    print(f"  {i}. Risk: {risk_id} <- Action: {action_id}")

Complex query: Risks mitigated by NIST AI RMF actions
Found 10 risk-action pairs:
  1. Risk: nist-information-security <- Action: MG-4.3-003
  2. Risk: nist-data-privacy <- Action: MG-4.3-003
  3. Risk: nist-information-integrity <- Action: MG-4.3-002


In [14]:
# This same query would be tedious with Python iteration:
# 1. Get all actions
# 2. Filter by NIST taxonomy
# 3. For each, look up related risks
# 4. Collect and deduplicate

# With OxigraphExplorer, it's a single declarative SPARQL query!
print("SPARQL is declarative and efficient for complex graph queries.")
print("OxigraphExplorer enables this without leaving the Pydantic object model.")

SPARQL is declarative and efficient for complex graph queries.
OxigraphExplorer enables this without leaving the Pydantic object model.


## Part 4: API Reference

In [15]:
ox = OxigraphExplorer(nexus._ontology)

# Detailed API Reference
methods_detailed = [
    {
        "name": "get_all_classes()",
        "description": "List all classes with data",
        "returns": "list[str]",
        "parameters": "None",
        "notes": "Derived from Container.model_fields_set"
    },
    {
        "name": "get_all(class_name, taxonomy, vocabulary, document)",
        "description": "Get all instances (supports collection keys like 'risks' or class names like 'Risk')",
        "returns": "list[Pydantic]",
        "parameters": "class_name (str | None), taxonomy (str | None, default: ibm-risk-atlas), vocabulary (str | None), document (str | None)",
        "notes": "Cached by parameter combination. Returns empty list if class not found."
    },
    {
        "name": "get_by_id(class_name, identifier)",
        "description": "Get a single instance by ID (O(1) lookup)",
        "returns": "Pydantic | None",
        "parameters": "class_name (str | None — unused), identifier (str)",
        "notes": "class_name parameter kept for API compatibility. Always checks id_cache."
    },
    {
        "name": "get_by_attribute(class_name, attribute, value)",
        "description": "Find instances by attribute value",
        "returns": "list[Pydantic]",
        "parameters": "class_name (str), attribute (str), value (Any)",
        "notes": "Uses SPARQL for efficient single-attribute filtering. Handles bool, int, float, str types."
    },
    {
        "name": "get_attribute(class_name, identifier, attribute)",
        "description": "Get a specific attribute from an instance",
        "returns": "Any | None",
        "parameters": "class_name (str | None), identifier (str), attribute (str)",
        "notes": "Convenience method combining get_by_id + getattr."
    },
    {
        "name": "query(class_name, **kwargs)",
        "description": "Query with keyword argument filters (AND logic)",
        "returns": "list[Pydantic]",
        "parameters": "class_name (str), **kwargs (attribute-value pairs)",
        "notes": "Pythonic alternative to filter_instances(). Equivalent to filter_instances(class_name, kwargs)."
    },
    {
        "name": "filter_instances(class_name, filters)",
        "description": "Filter by multiple attribute criteria (AND logic)",
        "returns": "list[Pydantic]",
        "parameters": "class_name (str), filters (dict)",
        "notes": "Fetches all instances, then filters in Python. Handles string and list attributes."
    },
    {
        "name": "filter_ids_by_type(ids, disallowed_types)",
        "description": "Filter ID list by type exclusion",
        "returns": "list[str]",
        "parameters": "ids (list[str]), disallowed_types (list[str])",
        "notes": "Keeps IDs NOT of specified types. Checks type of cached object."
    },
    {
        "name": "arrange_ids_by_type(ids)",
        "description": "Group IDs by their type",
        "returns": "dict[str, list[str]]",
        "parameters": "ids (list[str])",
        "notes": "Organizes IDs by object type name. Useful for type-aware batch processing."
    },
    {
        "name": "sparql_query(query_str)",
        "description": "Execute raw SPARQL query",
        "returns": "list[dict]",
        "parameters": "query_str (str)",
        "notes": "Returns raw SPARQL results (NamedNodes/Literals as strings). Prefix: 'nexus:' and 'rdf:'."
    },
    {
        "name": "object_similarity(entry_a, entry_b, threshold, property_scores, max_depth, weight_dict)",
        "description": "Measure similarity between two objects (0.0-100.0)",
        "returns": "float",
        "parameters": "entry_a (Any), entry_b (Any), threshold (float), property_scores (dict), max_depth (int=1), weight_dict (dict={})",
        "notes": "Updates property_scores dict in-place. Strings use semantic embeddings, numbers use exact match."
    },
    {
        "name": "object_equivalence(entry_a, entry_b, threshold, property_scores, max_depth, weight_dict)",
        "description": "Determine if objects are equivalent (threshold-based)",
        "returns": "bool",
        "parameters": "Same as object_similarity",
        "notes": "Returns True if object_similarity() >= threshold."
    },
    {
        "name": "clear_cache()",
        "description": "Clear query result cache",
        "returns": "None",
        "parameters": "None",
        "notes": "Clears _combined_cache only. ID cache persists (immutable). Does NOT clear embeddings."
    },
]

print("OxigraphExplorer Complete API Reference")
print("=" * 100)
print()

for i, method in enumerate(methods_detailed, 1):
    print(f"{i}. {method['name']}")
    print(f"   Description: {method['description']}")
    print(f"   Returns:     {method['returns']}")
    print(f"   Parameters:  {method['parameters']}")
    print(f"   Notes:       {method['notes']}")
    print()

OxigraphExplorer Complete API Reference

1. get_all_classes()
   Description: List all classes with data
   Returns:     list[str]
   Parameters:  None
   Notes:       Derived from Container.model_fields_set

2. get_all(class_name, taxonomy, vocabulary, document)
   Description: Get all instances (supports collection keys like 'risks' or class names like 'Risk')
   Returns:     list[Pydantic]
   Parameters:  class_name (str | None), taxonomy (str | None, default: ibm-risk-atlas), vocabulary (str | None), document (str | None)
   Notes:       Cached by parameter combination. Returns empty list if class not found.

3. get_by_id(class_name, identifier)
   Description: Get a single instance by ID (O(1) lookup)
   Returns:     Pydantic | None
   Parameters:  class_name (str | None — unused), identifier (str)
   Notes:       class_name parameter kept for API compatibility. Always checks id_cache.

4. get_by_attribute(class_name, attribute, value)
   Description: Find instances by attribute v

## Summary

`OxigraphExplorer` is a can be used alternative to `AtlasExplorer` that combines RDF graph capabilities with Pydantic object return types. This guide has covered:

**Core Functionality:**
- Basic queries: `get_all()`, `get_by_id()`, `get_by_attribute()`
- Multi-criteria filtering: `filter_instances()`, `query()` with kwargs
- Graph queries: `sparql_query()` for complex relationships
- Semantic similarity: `object_similarity()`, `object_equivalence()`

**Utility Methods:**
- `get_attribute()` — Retrieve single field from an object
- `filter_ids_by_type()` — Exclude IDs by type
- `arrange_ids_by_type()` — Organize IDs by entity type
- `clear_cache()` — Cache management

**Performance:**
- Query results are cached by parameter combination
- Use `get_by_id()` for fast O(1) lookups
- Use `get_by_attribute()` for single-attribute SPARQL filtering
- Use `sparql_query()` for complex relationship traversals
- Use `filter_instances()` or `query()` for multi-criteria AND filtering

**Use Cases:**

| Use Case | Method | Reason |
|----------|--------|--------|
| Get single object | `get_by_id()` | O(1) in-memory lookup |
| Get all of one type | `get_all()` | Cached, full table scan |
| Filter by one attribute | `get_by_attribute()` | SPARQL index query |
| Filter by multiple attributes | `filter_instances()` / `query()` | Python AND logic across fields |
| Find relationships | `sparql_query()` | Graph traversal, joins, patterns |
| Compare objects | `object_similarity()` | Semantic similarity with embeddings |
| Organize IDs by type | `arrange_ids_by_type()` | Batch processing by type |

## Part 5: Cache Management & Performance

OxigraphExplorer uses internal caching to optimize repeated queries. Understanding the caching behavior helps you write performant code:

## Part 6: Performance Considerations

When to use different query approaches for optimal performance:

In [16]:
print("""
QUERY STRATEGY GUIDE:
=====================

1. get_by_id(None, id) — O(1) lookup
   ✓ Use for: Single object lookups by ID
   ✓ Best for: Finding specific items when you know the ID
   Example: ox.get_by_id(None, "ibm-risk-atlas-001")

2. get_all(class_name) — Full table scan
   ✓ Use for: Retrieving all instances of a type
   ✓ Cached after first call
   ✓ Best for: Workloads that need many/all items from a class
   Example: ox.get_all("risks")

3. get_by_attribute(class_name, attr, value) — SPARQL index query
   ✓ Use for: Finding instances with specific attribute values
   ✓ Best for: Filtering on one attribute (single index)
   Example: ox.get_by_attribute("actions", "isDefinedByTaxonomy", "nist-ai-rmf")

4. filter_instances(class_name, filters) or query(class_name, **kwargs) — Python filter
   ✓ Use for: Multi-criteria AND filtering
   ✓ Best for: Complex conditions across multiple attributes
   ✓ Note: Fetches all instances, then filters in Python
   Example: ox.filter_instances("risks", {"isDefinedByTaxonomy": "ibm", "descriptor": "attack"})

5. sparql_query(query_str) — Custom graph traversal
   ✓ Use for: Complex relationships (joins, paths, aggregations)
   ✓ Best for: Finding connected entities, relationship patterns
   ✓ Most powerful but requires SPARQL knowledge
   Example: ox.sparql_query("SELECT ?risk WHERE {?action nexus:hasRelatedRisk ?risk}")

CACHING BEHAVIOR:
=================

• Query Results: Cached by (class_names, taxonomies, vocabulary, document)
  - Cache key includes ALL parameters
  - Different parameters = new cache entry
  - Identical parameters = instant cache hit

• ID Lookups: NOT cached (already O(1) in-memory dict)
  - get_by_id() always hits the id_cache directly
  - No cache invalidation needed for lookups

• When to call clear_cache():
  - If the underlying data changed (loaded new ontology)
  - To free memory if OxigraphExplorer is long-lived
  - Before/after benchmarking query performance
  - Note: ID cache persists (needed for proper object resolution)
""")


QUERY STRATEGY GUIDE:

1. get_by_id(None, id) — O(1) lookup
   ✓ Use for: Single object lookups by ID
   ✓ Best for: Finding specific items when you know the ID
   Example: ox.get_by_id(None, "ibm-risk-atlas-001")

2. get_all(class_name) — Full table scan
   ✓ Use for: Retrieving all instances of a type
   ✓ Cached after first call
   ✓ Best for: Workloads that need many/all items from a class
   Example: ox.get_all("risks")

3. get_by_attribute(class_name, attr, value) — SPARQL index query
   ✓ Use for: Finding instances with specific attribute values
   ✓ Best for: Filtering on one attribute (single index)
   Example: ox.get_by_attribute("actions", "isDefinedByTaxonomy", "nist-ai-rmf")

4. filter_instances(class_name, filters) or query(class_name, **kwargs) — Python filter
   ✓ Use for: Multi-criteria AND filtering
   ✓ Best for: Complex conditions across multiple attributes
   ✓ Note: Fetches all instances, then filters in Python
   Example: ox.filter_instances("risks", {"isDefined

In [17]:
print("""
=== object_similarity ===
Measure how similar two objects are by comparing all fields.

Signature:
  object_similarity(entry_a, entry_b, threshold, property_scores, max_depth=1, weight_dict={})

Args:
  - entry_a, entry_b (Any): Pydantic objects to compare
  - threshold (float): Minimum score for nested object equivalence (0.0-100.0)
  - property_scores (dict): Mutable dict populated with per-field scores and aggregates
  - max_depth (int): Recursion depth for comparing referenced objects (default 1)
  - weight_dict (dict): Custom field weights; weight=0 excludes field (default equal)

Returns:
  float: Similarity score 0.0-100.0

Field comparison strategies:
  - Strings: Semantic embedding similarity (via txtai)
  - ID references at depth > 0: Recursive object_similarity
  - Booleans, ints, floats: Exact match (100.0 or 0.0)
  - Lists: Jaccard similarity (intersection/union)
  - None values: Both None=100%, one None=0%

property_scores dict structure:
  {
    "field_name": {
      "score": 95.5,              # Score for this field (0.0-100.0)
      "weight": 1.0,              # Weight applied to this field
      "contributing_score": 95.5, # score * weight
    },
    "_sum_of_weights": 42.0,      # Total weight across scored fields
    "_final_score": 87.3,         # Weighted average across all fields
  }

---
=== object_equivalence ===
Determine if two objects are semantically equivalent based on threshold.

Signature:
  object_equivalence(entry_a, entry_b, threshold, property_scores, max_depth=1, weight_dict={})

Args:
  Same as object_similarity

Returns:
  bool: True if similarity >= threshold, False otherwise
""")


=== object_similarity ===
Measure how similar two objects are by comparing all fields.

Signature:
  object_similarity(entry_a, entry_b, threshold, property_scores, max_depth=1, weight_dict={})

Args:
  - entry_a, entry_b (Any): Pydantic objects to compare
  - threshold (float): Minimum score for nested object equivalence (0.0-100.0)
  - property_scores (dict): Mutable dict populated with per-field scores and aggregates
  - max_depth (int): Recursion depth for comparing referenced objects (default 1)
  - weight_dict (dict): Custom field weights; weight=0 excludes field (default equal)

Returns:
  float: Similarity score 0.0-100.0

Field comparison strategies:
  - Strings: Semantic embedding similarity (via txtai)
  - ID references at depth > 0: Recursive object_similarity
  - Booleans, ints, floats: Exact match (100.0 or 0.0)
  - Lists: Jaccard similarity (intersection/union)
  - None values: Both None=100%, one None=0%

property_scores dict structure:
  {
    "field_name": {
      "s

## Similarity API Reference

In [18]:
# Find potentially duplicate risks (similarity >= 85%) across taxonomies
all_risks = ox.get_all("risks")
duplicate_pairs = []

# Compare the IBM AI Atlas risk "atlas-harmful-output" against all others
reference_risk = ox.get_by_id(None, "atlas-harmful-output")
threshold = 85.0

print(f"Scanning {len(all_risks)} risks for possible duplicates of {reference_risk.id}...")
print(f"Threshold: {threshold}%")
print()

for other_risk in all_risks:
    if other_risk.id == reference_risk.id:
        continue
    
    scores = {}
    similarity = ox.object_similarity(
        reference_risk, 
        other_risk, 
        threshold=threshold,
        property_scores=scores
    )
    
    if similarity >= threshold:
        duplicate_pairs.append((other_risk.id, similarity))

if duplicate_pairs:
    print(f"Found {len(duplicate_pairs)} potential duplicates:\n")
    for risk_id, sim in sorted(duplicate_pairs, key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {risk_id:30} | Similarity: {sim:.2f}%")
else:
    print(f"No duplicates found at {threshold}% threshold")

Scanning 546 risks for possible duplicates of atlas-harmful-output...
Threshold: 85.0%



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11290.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


No duplicates found at 85.0% threshold


### Practical Use Cases

**Finding duplicate or near-duplicate risks across taxonomies:**

In [19]:
# Compare at different depths
risks = ox.get_all("risks")[:5]  # Use a sample of risks
r1, r2 = risks[0], risks[1]

# Depth 0: only direct field comparison (no recursion into referenced objects)
scores_d0 = {}
sim_d0 = ox.object_similarity(
    r1, r2,
    threshold=50.0,
    property_scores=scores_d0,
    max_depth=0
)

# Depth 1: include comparison of objects referenced by ID fields
scores_d1 = {}
sim_d1 = ox.object_similarity(
    r1, r2,
    threshold=50.0,
    property_scores=scores_d1,
    max_depth=1
)

print(f"Similarity comparison at different recursion depths:")
print(f"  {r1.id} vs {r2.id}")
print()
print(f"  max_depth=0 (direct fields only):     {sim_d0:.2f}%")
print(f"  max_depth=1 (include references):     {sim_d1:.2f}%")
print()
print("Note: Depth=1 can find higher similarity if objects share referenced entities")
print("      (e.g., risks from the same taxonomy, group, or with shared mappings)")

Similarity comparison at different recursion depths:
  atlas-evasion-attack vs atlas-impact-on-the-environment

  max_depth=0 (direct fields only):     58.73%
  max_depth=1 (include references):     61.42%

Note: Depth=1 can find higher similarity if objects share referenced entities
      (e.g., risks from the same taxonomy, group, or with shared mappings)


### Recursion Depth: Compare Referenced Objects

With `max_depth > 0`, similarity includes comparison of objects referenced by ID fields:

In [20]:
# Default: all fields equally weighted
scores_default = {}
sim_default = ox.object_similarity(
    r1, r2, 
    threshold=50.0, 
    property_scores=scores_default
)

print(f"Similarity (default weights): {sim_default:.2f}%")
print()

# Custom: prioritize name and description, ignore id and url
weight_dict = {
    "name": 10.0,          # High weight — very important
    "description": 5.0,    # Medium weight
    "id": 0,               # Exclude id from comparison
    "url": 0,              # Exclude url
}

scores_weighted = {}
sim_weighted = ox.object_similarity(
    r1, r2,
    threshold=50.0,
    property_scores=scores_weighted,
    weight_dict=weight_dict
)

print(f"Similarity (custom weights):")
print(f"  - name weight = 10.0 (high priority)")
print(f"  - description weight = 5.0 (medium)")
print(f"  - id, url excluded (weight = 0)")
print(f"  Result: {sim_weighted:.2f}%")
print()

# Show fields that were scored
weighted_fields = [f for f in scores_weighted if not f.startswith('_')]
print(f"Fields scored with custom weights: {len(weighted_fields)}")
print(f"'id' in scores: {'id' in scores_weighted}")
print(f"'name' in scores: {'name' in scores_weighted}")

Similarity (default weights): 61.42%

Similarity (custom weights):
  - name weight = 10.0 (high priority)
  - description weight = 5.0 (medium)
  - id, url excluded (weight = 0)
  Result: 45.54%

Fields scored with custom weights: 24
'id' in scores: False
'name' in scores: True


### Custom Weights: Prioritize Important Fields

Control which fields matter most by providing custom weights:

In [21]:
# Test equivalence at different thresholds
risks = ox.get_all("risks")
risk_a, risk_b = risks[0], risks[1]

scores_eq = {}
sim = ox.object_similarity(risk_a, risk_b, threshold=50.0, property_scores=scores_eq)

print(f"Similarity between {risk_a.id} and {risk_b.id}: {sim:.2f}%")
print()

# Test equivalence at different thresholds
thresholds = [50.0, 75.0, 90.0, 95.0]
print("Equivalence at different thresholds:")

for threshold in thresholds:
    equiv = ox.object_equivalence(risk_a, risk_b, threshold=threshold, property_scores={})
    status = "✓ EQUIVALENT" if equiv else "✗ NOT equivalent"
    print(f"  Threshold {threshold:5.1f}%: {status}")

Similarity between atlas-evasion-attack and atlas-impact-on-the-environment: 61.42%

Equivalence at different thresholds:
  Threshold  50.0%: ✓ EQUIVALENT
  Threshold  75.0%: ✗ NOT equivalent
  Threshold  90.0%: ✗ NOT equivalent
  Threshold  95.0%: ✗ NOT equivalent


### Equivalence: Threshold-Based Classification

Use `object_equivalence()` to classify whether two objects are "equivalent" based on a similarity threshold:

In [22]:
# Inspect individual field scores from the comparison
scores_detail = {}
ox.object_similarity(r1, r2, threshold=0.5, property_scores=scores_detail)

print("Field-by-field similarity scores:")
print("Field            | Score  | Weight | Contributing")
print("-" * 50)

field_scores = [(k, v) for k, v in scores_detail.items() if not k.startswith('_')]
for field_name, field_data in sorted(field_scores)[:10]:
    score = field_data["score"]
    weight = field_data["weight"]
    contrib = field_data["contributing_score"]
    print(f"{field_name:16} | {score:6.1f} | {weight:6.1f} | {contrib:6.1f}")

Field-by-field similarity scores:
Field            | Score  | Weight | Contributing
--------------------------------------------------
broad_mappings   |    0.0 |    1.0 |    0.0
close_mappings   |    0.0 |    1.0 |    0.0
concern          |   14.9 |    1.0 |   14.9
dateCreated      |    0.0 |    1.0 |    0.0
dateModified     |    0.0 |    1.0 |    0.0
description      |   24.4 |    1.0 |   24.4
descriptor       |  100.0 |    1.0 |  100.0
detectsRiskConcept |  100.0 |    1.0 |  100.0
exact_mappings   |    0.0 |    1.0 |    0.0
hasDocumentation |  100.0 |    1.0 |  100.0


### Inspecting Per-Field Scores

The `property_scores` dict shows you how similarity was computed across all fields:

In [23]:
# Compare an object to itself (should be 100% similar)
scores = {}
similarity_same = ox.object_similarity(r1, r1, threshold=0.5, property_scores=scores)

print(f"Similarity (risk1 vs risk1):")
print(f"  Score: {similarity_same:.2f}%")
print(f"  Final score: {scores['_final_score']:.2f}%")
print(f"  Sum of weights: {scores['_sum_of_weights']:.1f}")
print()

# Compare two different risks
scores2 = {}
similarity_diff = ox.object_similarity(r1, r2, threshold=0.5, property_scores=scores2)

print(f"Similarity (risk1 vs risk2):")
print(f"  Score: {similarity_diff:.2f}%")
print(f"  Fields scored: {len([f for f in scores2 if not f.startswith('_')])}")

Similarity (risk1 vs risk1):
  Score: 92.31%
  Final score: 92.31%
  Sum of weights: 26.0

Similarity (risk1 vs risk2):
  Score: 61.42%
  Fields scored: 26


### Basic Similarity Comparison

Compare two objects and get a similarity score (0.0–100.0):

In [24]:
# Get two risks to compare
risks = ox.get_all("risks")
risk1, risk2 = risks[0], risks[1]

print(f"Comparing two risks:")
print(f"  Risk 1: {risk1.id} - {risk1.name}")
print(f"  Risk 2: {risk2.id} - {risk2.name}")

Comparing two risks:
  Risk 1: atlas-evasion-attack - Evasion attack
  Risk 2: atlas-impact-on-the-environment - Impact on the environment


## Part 5: Semantic Similarity — Comparing Ontology Entries

OxigraphExplorer now includes powerful methods for measuring how similar two ontology entries are:

## Part 7: Star Graph & Subgraph Comparison

OxigraphExplorer provides methods to extract and compare star graphs (1-hop neighborhoods) of entities in the knowledge graph. This is useful for understanding how two risks or actions relate to overlapping sets of neighbors.

In [25]:
# Build a star graph for the first risk
risk_id_a = risks[0].id
star_graph_a = ox.get_star_graph(risk_id_a)

print(f"Star Graph for {risk_id_a}:")
print(f"  Center: {star_graph_a.center.name}")
print(f"  Total neighbors: {len(star_graph_a.neighbor_ids)}")
print()
print(f"  Neighbors by relationship type:")
for field_name, neighbors in star_graph_a.neighbors.items():
    print(f"    {field_name:20} : {len(neighbors):3} neighbors")

Star Graph for atlas-evasion-attack:
  Center: Evasion attack
  Total neighbors: 10

  Neighbors by relationship type:
    close_mappings       :   2 neighbors
    related_mappings     :   5 neighbors
    broad_mappings       :   1 neighbors
    isDefinedByTaxonomy  :   1 neighbors
    isPartOf             :   1 neighbors


In [26]:
# Build a star graph for a second risk
risk_id_b = risks[1].id
star_graph_b = ox.get_star_graph(risk_id_b)

print(f"Star Graph for {risk_id_b}:")
print(f"  Center: {star_graph_b.center.name}")
print(f"  Total neighbors: {len(star_graph_b.neighbor_ids)}")
print()
print(f"  Neighbors by relationship type:")
for field_name, neighbors in star_graph_b.neighbors.items():
    print(f"    {field_name:20} : {len(neighbors):3} neighbors")

Star Graph for atlas-impact-on-the-environment:
  Center: Impact on the environment
  Total neighbors: 8

  Neighbors by relationship type:
    exact_mappings       :   1 neighbors
    related_mappings     :   5 neighbors
    isDefinedByTaxonomy  :   1 neighbors
    isPartOf             :   1 neighbors


In [27]:
# Compare the two star graphs
comparison = ox.compare_star_graphs(risk_id_a, risk_id_b)

print(f"Star Graph Comparison: {risk_id_a} vs {risk_id_b}")
print()
print(f"  Jaccard Similarity: {comparison.jaccard_similarity:.4f} ({comparison.jaccard_similarity*100:.2f}%)")
print()
print(f"  Shared neighbors: {len(comparison.shared_neighbor_ids)}")
if comparison.shared_neighbor_ids:
    print(f"    Examples: {', '.join(list(comparison.shared_neighbor_ids)[:3])}")
print()
print(f"  Unique to {risk_id_a}: {len(comparison.unique_to_a)}")
if comparison.unique_to_a:
    print(f"    Examples: {', '.join(list(comparison.unique_to_a)[:3])}")
print()
print(f"  Unique to {risk_id_b}: {len(comparison.unique_to_b)}")
if comparison.unique_to_b:
    print(f"    Examples: {', '.join(list(comparison.unique_to_b)[:3])}")

Star Graph Comparison: atlas-evasion-attack vs atlas-impact-on-the-environment

  Jaccard Similarity: 0.1250 (12.50%)

  Shared neighbors: 2
    Examples: mit-ai-causal-risk-timing-post-deployment, ibm-risk-atlas

  Unique to atlas-evasion-attack: 8
    Examples: aiuc1-req-b002, ibm-risk-atlas-robustness-model-behavior-manipulation, aiuc1-req-b001

  Unique to atlas-impact-on-the-environment: 6
    Examples: nist-environmental-impacts, ibm-risk-atlas-societal-impact, mit-ai-causal-risk-intent-other


In [28]:
# Per-field breakdown: how neighbors differ by relationship type
print("\nPer-field breakdown (shared/unique neighbors by relationship type):")
print("=" * 80)
for field_name, breakdown in comparison.per_field.items():
    shared_count = len(breakdown['shared'])
    unique_a_count = len(breakdown['unique_to_a'])
    unique_b_count = len(breakdown['unique_to_b'])
    if shared_count > 0 or unique_a_count > 0 or unique_b_count > 0:
        print(f"\n{field_name}:")
        print(f"  Shared:     {shared_count}")
        print(f"  Only in A:  {unique_a_count}")
        print(f"  Only in B:  {unique_b_count}")


Per-field breakdown (shared/unique neighbors by relationship type):

broad_mappings:
  Shared:     0
  Only in A:  1
  Only in B:  0

close_mappings:
  Shared:     0
  Only in A:  2
  Only in B:  0

isDefinedByTaxonomy:
  Shared:     1
  Only in A:  0
  Only in B:  0

related_mappings:
  Shared:     1
  Only in A:  4
  Only in B:  4

isPartOf:
  Shared:     0
  Only in A:  1
  Only in B:  1

exact_mappings:
  Shared:     0
  Only in A:  0
  Only in B:  1
